# 🌿 CropVision Training (Pure Vision Mode)

This notebook trains a crop disease classification model using **EfficientNet-B3**. It has been simplified to use **images only**, removing the complex VLM components to ensure stability and accuracy.

In [ ]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
else:
    print('⚠️ No GPU! Go to Runtime > Change runtime type > T4 GPU')

!pip install -q scikit-learn seaborn matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

In [ ]:
import os, zipfile, shutil

# ✏️ Change these paths to your Google Drive location
DATASET_ZIP_PATH = '/content/drive/MyDrive/crop/dataset_final.zip'
EXTRACT_DIR      = '/content/dataset'

if not os.path.exists(EXTRACT_DIR) or len(os.listdir(EXTRACT_DIR)) < 3:
    print('Extracting dataset...')
    with zipfile.ZipFile(DATASET_ZIP_PATH, 'r') as z:
        z.extractall('/content/dataset/')
    print('Done!')
else:
    print('Dataset already extracted.')

# 🏁 Verification of Splits
for split in ['train', 'val', 'test']:
    path = os.path.join(EXTRACT_DIR, split)
    if os.path.exists(path):
        classes = [d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))]
        total = sum(len(os.listdir(os.path.join(path, c))) for c in classes)
        print(f'{split}: {len(classes)} classes, {total} images')
    else:
        print(f'⚠️ {split} folder not found at {path}')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
import json, os, numpy as np
import time, copy
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

CONFIG = {
    'train_dir' : '/content/dataset/train',
    'val_dir'   : '/content/dataset/val',
    'test_dir'  : '/content/dataset/test',
    'img_size'  : 300,
    'batch_size': 32,
    'lr'        : 1e-3,
    'epochs'    : 20,
    'num_workers': 2,
    'save_path' : '/content/drive/MyDrive/crop/best_model_vision.pth',
    'device'    : 'cuda' if torch.cuda.is_available() else 'cpu'
}

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(CONFIG['img_size']),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(CONFIG['img_size'] + 20),
    transforms.CenterCrop(CONFIG['img_size']),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
train_ds_folder = datasets.ImageFolder(CONFIG['train_dir'])
train_classes = train_ds_folder.classes
num_classes   = len(train_classes)
print(f'Classes: {train_classes}')

counts = np.array([len(os.listdir(os.path.join(CONFIG['train_dir'], c))) for c in train_classes])
class_weights_arr = counts.sum() / (num_classes * counts)
sample_weights = [class_weights_arr[label] for _, label in train_ds_folder.samples]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(datasets.ImageFolder(CONFIG['train_dir'], train_transforms), batch_size=CONFIG['batch_size'], sampler=sampler)
val_loader   = DataLoader(datasets.ImageFolder(CONFIG['val_dir'], val_transforms),   batch_size=CONFIG['batch_size'], shuffle=False)
test_loader  = DataLoader(datasets.ImageFolder(CONFIG['test_dir'], val_transforms),  batch_size=CONFIG['batch_size'], shuffle=False)

model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
model = model.to(CONFIG['device'])

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])

In [ ]:
best_acc = 0.0
since = time.time()

for epoch in range(CONFIG['epochs']):
    print(f'\nEpoch {epoch+1}/{CONFIG["epochs"]}')
    
    for phase in ['train', 'val']:
        model.train() if phase == 'train' else model.eval()
        running_loss, running_corrects = 0.0, 0
        loader = train_loader if phase == 'train' else val_loader
        
        for inputs, labels in loader:
            inputs, labels = inputs.to(CONFIG['device']), labels.to(CONFIG['device'])
            optimizer.zero_grad()
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)
                if phase == 'train':
                    loss.backward()
                    optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data).item()
            
        epoch_loss = running_loss / len(loader.dataset)
        epoch_acc = running_corrects / len(loader.dataset)
        print(f'{phase:5s} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
        
        if phase == 'val' and epoch_acc > best_acc:
            best_acc = epoch_acc
            torch.save(model.state_dict(), CONFIG['save_path'])
            print(f'✅ Best model saved to Drive! Accuracy: {best_acc:.4f}')
            
    scheduler.step()

In [ ]:
model.load_state_dict(torch.load(CONFIG['save_path']))
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(CONFIG['device'])
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print('\n=== Classification Report ===')
print(classification_report(all_labels, all_preds, target_names=train_classes))